# CXR-MultiQuant — Phase 2: Model Architecture
Building the Multimodal deep learning architecture using **TensorFlow / Keras**.

- **Vision**: DenseNet-121
- **Text**: ClinicalBERT
- **Fusion**: Co-Attention
- **Loss**: Focal Loss

In [ ]:

!pip install tensorflow transformers datasets pandas matplotlib scikit-learn

In [ ]:
# Cell 2: Imports and Setup
import os
import io
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoTokenizer, TFAutoModel

from datasets import load_dataset
from PIL import Image

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 3: Load Data and Tokenizer
print("Loading Hugging Face dataset...")
hf_dataset = load_dataset("itsanmolgupta/mimic-cxr-dataset", split='train')
hf_df = hf_dataset.to_pandas()

print("Loading generated CSV labels...")
train_csv = pd.read_csv('train_labels.csv')
val_csv = pd.read_csv('val_labels.csv')
test_csv = pd.read_csv('test_labels.csv')

# Merge CSVs back with the Hugging Face dataset to get the images
train_df = pd.merge(hf_df, train_csv[['impression', 'severity']], on='impression', how='inner')
val_df = pd.merge(hf_df, val_csv[['impression', 'severity']], on='impression', how='inner')
test_df = pd.merge(hf_df, test_csv[['impression', 'severity']], on='impression', how='inner')

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

# Load ClinicalBERT Tokenizer
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
MAX_LEN = 128  # Max token length for the reports

In [ ]:
# Cell 4: Create tf.data.Dataset Generator
label_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}

def extract_image(img_data):
    if isinstance(img_data, dict) and 'bytes' in img_data:
        img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
    else:
        img = img_data.convert('RGB')
    
    img = img.resize((224, 224))
    return np.array(img) / 255.0  # Normalize to [0, 1]

def data_generator(df):
    for idx, row in df.iterrows():
        img_array = extract_image(row['image'])
        
        tokens = tokenizer(
            str(row['impression']),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors="tf"
        )
        input_ids = tf.squeeze(tokens['input_ids'])
        attention_mask = tf.squeeze(tokens['attention_mask'])
        
        label = label_map[row['severity']]
        yield (img_array, input_ids, attention_mask), label

output_signature = (
    (
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),  # Image
        tf.TensorSpec(shape=(MAX_LEN,), dtype=tf.int32),       # Input IDs
        tf.TensorSpec(shape=(MAX_LEN,), dtype=tf.int32)        # Attention Mask
    ),
    tf.TensorSpec(shape=(), dtype=tf.int32)                    # Label
)

BATCH_SIZE = 16

train_ds = tf.data.Dataset.from_generator(lambda: data_generator(train_df), output_signature=output_signature)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(lambda: data_generator(val_df), output_signature=output_signature)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("tf.data Pipelines created successfully!")

## 1. Image Encoder (DenseNet-121)
Pretrained on ImageNet, augmented, and projected to 256 dimensions.

In [ ]:
# Cell 5: Image Encoder
def create_image_encoder():
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    
    # Data Augmentation
    x = layers.RandomRotation(0.1)(image_input)
    x = layers.RandomZoom(0.1)(x)
    
    # Pretrained DenseNet121
    densenet = keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    densenet.trainable = False  # Freeze for initial training
    
    x = densenet.output
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense Projection Layer -> 256-dim
    img_features = layers.Dense(256, activation='relu', name="img_projection")(x)
    
    return keras.Model(inputs=image_input, outputs=img_features, name="Image_Encoder")

image_encoder = create_image_encoder()
image_encoder.summary()

## 2. Text Encoder (ClinicalBERT)
Extracts the [CLS] token and projects to 256 dimensions.

In [ ]:
# Cell 6: Text Encoder
def create_text_encoder():
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")
    
    # Load Hugging Face TF ClinicalBERT
    clinical_bert = TFAutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT", from_pt=True)
    clinical_bert.trainable = False  # Freeze for initial training
    
    bert_output = clinical_bert(input_ids, attention_mask=attention_mask)
    
    # Extract [CLS] Token (first token in sequence)
    cls_token = bert_output.last_hidden_state[:, 0, :]
    
    # Dense Projection Layer -> 256-dim
    txt_features = layers.Dense(256, activation='relu', name="txt_projection")(cls_token)
    
    return keras.Model(inputs=[input_ids, attention_mask], outputs=txt_features, name="Text_Encoder")

text_encoder = create_text_encoder()
text_encoder.summary()

## 3. Co-Attention Fusion & Classifier
Cross attention between Image and Text, followed by the severity classifier.

In [ ]:
# Cell 7: Full Multimodal Model with Co-Attention
def create_multimodal_model():
    # Inputs
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")
    
    # Encoders
    img_feats = image_encoder(image_input)                 # (batch, 256)
    txt_feats = text_encoder([input_ids, attention_mask])  # (batch, 256)
    
    # To use MultiHeadAttention, we need 3D tensors: (batch_size, seq_len, dim)
    img_feats_seq = tf.expand_dims(img_feats, axis=1)      # (batch, 1, 256)
    txt_feats_seq = tf.expand_dims(txt_feats, axis=1)      # (batch, 1, 256)
    
    # --- CO-ATTENTION FUSION ---
    # Cross 1: Image -> Text (Q: Image, K/V: Text)
    attn_img_txt = layers.MultiHeadAttention(num_heads=8, key_dim=32, name="cross1_img_txt")(
        query=img_feats_seq, value=txt_feats_seq, key=txt_feats_seq
    )
    # Residual & LayerNorm
    norm1 = layers.LayerNormalization()(img_feats_seq + attn_img_txt)
    
    # Cross 2: Text -> Image (Q: Text, K/V: Image)
    attn_txt_img = layers.MultiHeadAttention(num_heads=8, key_dim=32, name="cross2_txt_img")(
        query=txt_feats_seq, value=img_feats_seq, key=img_feats_seq
    )
    # Residual & LayerNorm
    norm2 = layers.LayerNormalization()(txt_feats_seq + attn_txt_img)
    
    # Squeeze back to 2D and Concatenate -> 512-dim
    norm1_flat = tf.squeeze(norm1, axis=1)
    norm2_flat = tf.squeeze(norm2, axis=1)
    fused = layers.Concatenate(name="fused_concat")([norm1_flat, norm2_flat])
    
    # --- CLASSIFIER HEAD ---
    x = layers.Dense(256, activation='relu', name="fc1")(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='relu', name="fc2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    output = layers.Dense(3, activation='softmax', name="severity_prediction")(x)
    
    return keras.Model(inputs=[image_input, input_ids, attention_mask], outputs=output, name="CXR_MultiQuant")

multimodal_model = create_multimodal_model()
multimodal_model.summary()

## 4. Compile with Focal Loss

In [ ]:
# Cell 8: Define Focal Loss and Compile Model
class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # y_true is passed as integer labels from the dataset (sparse)
        # We need to one-hot encode them first
        y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=3)
        
        y_pred = tf.clip_by_value(y_pred, keras.backend.epsilon(), 1.0 - keras.backend.epsilon())
        cross_entropy = -y_true_one_hot * tf.math.log(y_pred)
        loss = self.alpha * tf.math.pow(1 - y_pred, self.gamma) * cross_entropy
        return tf.reduce_sum(loss, axis=-1)

multimodal_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=['accuracy']
)

print("Model compiled successfully with Focal Loss!")